# Maze Escape — Charger un modèle entraîné sur votre propre map

Ce notebook sert à :

1. **charger un checkpoint déjà entraîné**  
2. **remplacer la map par votre `CUSTOM_MAP`**  
3. **visualiser l'agent dans une fenêtre Pygame**  
4. tester plusieurs stratégies :
   - `sample` 
   - `greedy`
   - `temperature`

**Important**
- Ce notebook est conçu pour le **modèle 16×16 full-map**.
- Votre map personnalisée doit donc être une **grille 16×16**.
- Les codes de cellules doivent être :
  - `0 = free`
  - `1 = wall`
  - `2 = trap`
  - `3 = exit`
  - `5 = coin`
  - `6 = enemy`
- Le joueur n'a pas besoin d'être écrit dans la map : il démarre en `(0, 0)`.


In [36]:
# Dépendances (à exécuter une seule fois si besoin)
# %pip install pygame torch matplotlib numpy


In [37]:
import os
import time
import random
from typing import Dict, Optional, Tuple, List

import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical

try:
    import pygame
    PYGAME_AVAILABLE = True
except Exception as e:
    pygame = None
    PYGAME_AVAILABLE = False
    print("[WARN] pygame import failed:", e)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[INFO] device =", DEVICE)
print("[INFO] pygame available =", PYGAME_AVAILABLE)


[INFO] device = cpu
[INFO] pygame available = True


In [50]:
# Modèle à utiliser
MODEL_FILENAME = "maze_policy_v5_greedy_finetuned_best_sample.pth"

# Le notebook va chercher automatiquement le modèle ici :
# - dans le dossier courant
# - dans le même dossier que le notebook
# - dans /mnt/data (utile dans ChatGPT / sandbox)
CHECKPOINT_CANDIDATES = [
    MODEL_FILENAME,
    f"./{MODEL_FILENAME}",
    f"/mnt/data/{MODEL_FILENAME}",
]

def resolve_checkpoint_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        "Checkpoint introuvable. Vérifie que le fichier .pth est dans le même dossier que le notebook "
        "ou modifie CHECKPOINT_CANDIDATES."
    )

CHECKPOINT_PATH = resolve_checkpoint_path(CHECKPOINT_CANDIDATES)

# Choix par défaut
PLAY_STRATEGY = "sample"   # "sample", "greedy", "temperature"
TEMPERATURE = 0.30
SELECTED_LEVEL = "easy"    # "easy", "medium", "hard"

CELL_SIZE = 42
FPS = 8
MAX_STEPS = 200

# Place tes sprites dans le même dossier que le notebook
# (ou remplace simplement les chemins par tes chemins absolus Windows).
ASSET_PATHS = {
    "player": "./player.png",
    "wall":   "./wall.png",
    "exit":   "./exit.png",
    "trap":   "./trap.png",
    "coin":   "./coin.png",
    "free":   "./floor.png",
    # "enemy": "./enemy.png",  # optionnel
}

print("[OK] modèle sélectionné :", CHECKPOINT_PATH)
print("[OK] niveau sélectionné par défaut :", SELECTED_LEVEL)

[OK] modèle sélectionné : maze_policy_v5_greedy_finetuned_best_sample.pth
[OK] niveau sélectionné par défaut : easy


In [63]:
LEVEL_MAPS = {
    "easy": [
        [0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
        [1,0,0,0,0,1,0,0,0,0,0,0,5,0,0,1],
        [1,1,1,1,0,1,0,1,1,1,1,1,1,1,0,1],
        [1,0,0,1,0,1,0,1,0,0,0,0,0,1,0,1],
        [1,0,0,1,0,1,0,1,0,1,1,1,0,1,0,1],
        [1,0,1,1,0,0,0,1,0,0,5,1,0,0,0,1],
        [1,0,1,2,1,1,0,1,1,1,0,1,1,1,0,1],
        [1,0,1,0,0,0,0,0,0,1,0,0,0,1,0,1],
        [1,0,1,0,1,1,1,1,0,1,1,1,0,1,0,1],
        [1,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1],
        [1,1,1,0,1,1,0,1,1,1,0,1,1,1,0,1],
        [1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,1],
        [1,0,1,1,1,1,1,1,0,1,1,1,0,1,0,1],
        [1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1],
        [1,1,1,1,1,0,1,1,1,1,1,1,0,0,0,0],
        [1,1,1,1,1,0,0,0,0,2,0,1,1,1,1,3],
    ],
    "medium": [
        [0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
        [1,0,0,0,0,0,0,0,0,1,0,5,0,0,0,1],
        [1,1,1,1,1,0,1,1,0,1,0,1,1,1,0,1],
        [1,0,0,0,1,0,1,0,0,0,0,1,2,1,0,1],
        [1,0,1,1,1,0,1,0,1,1,0,1,0,1,0,1],
        [1,0,0,0,0,0,1,0,1,5,0,0,0,1,0,1],
        [1,1,1,1,1,0,1,0,1,1,1,1,0,1,0,1],
        [1,0,0,0,1,0,0,0,0,0,2,1,0,0,0,1],
        [1,0,1,0,1,1,1,1,1,0,1,1,1,1,0,1],
        [1,0,1,0,0,0,0,0,1,0,1,5,0,1,0,1],
        [1,0,1,1,1,1,1,0,1,0,1,1,0,1,0,1],
        [1,0,0,0,0,2,1,0,0,0,0,1,0,0,0,1],
        [1,1,1,1,0,1,1,1,1,1,0,1,1,1,0,1],
        [1,0,5,0,0,0,0,0,0,1,0,0,0,1,0,1],
        [1,0,1,1,1,1,1,1,0,1,1,1,0,1,0,0],
        [1,0,0,0,0,0,0,1,0,0,0,5,0,0,0,3],
    ],
    "hard": [
        [0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
        [1,0,1,0,0,0,1,0,0,0,0,1,5,0,0,1],
        [1,0,1,0,1,0,1,0,1,1,0,1,1,1,0,1],
        [1,0,0,1,1,0,0,0,1,0,0,0,2,1,0,1],
        [1,1,0,1,1,1,1,0,1,0,1,1,0,1,0,1],
        [1,0,0,0,0,0,1,0,1,0,0,1,0,0,0,1],
        [1,0,1,1,1,0,1,2,1,1,0,1,1,1,0,1],
        [1,0,0,0,1,0,0,0,0,1,0,0,5,1,0,1],
        [1,1,1,0,1,1,1,1,0,1,1,1,0,1,0,1],
        [1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1],
        [1,0,1,1,1,0,1,1,1,1,1,1,1,1,0,1],
        [1,0,0,0,0,0,1,0,0,0,0,1,0,2,0,1],
        [1,1,1,1,1,0,1,0,1,1,1,1,0,1,0,1],
        [1,0,5,0,0,0,1,0,0,0,0,0,0,1,0,1],
        [1,0,1,1,1,1,1,1,1,1,1,1,0,1,0,0],
        [1,0,0,0,0,0,0,0,0,5,0,0,0,2,0,3],
    ],
}

LEVEL_ORDER = ["easy", "medium", "hard"]

def get_level_map(level_name: str) -> np.ndarray:
    if level_name not in LEVEL_MAPS:
        raise ValueError(f"Niveau inconnu: {level_name}. Choisis parmi {list(LEVEL_MAPS)}")
    return np.array(LEVEL_MAPS[level_name], dtype=np.int64)

CUSTOM_MAP = LEVEL_MAPS[SELECTED_LEVEL]
print("[OK] niveaux disponibles :", LEVEL_ORDER)
print("[OK] niveau actif :", SELECTED_LEVEL)

[OK] niveaux disponibles : ['easy', 'medium', 'hard']
[OK] niveau actif : easy


In [40]:
GRID_SIZE = 16
CELL_FREE = 0
CELL_WALL = 1
CELL_TRAP = 2
CELL_EXIT = 3
CELL_PLAYER = 4
CELL_COIN = 5
CELL_ENEMY = 6
N_CELL_TYPES = 7

for level_name in LEVEL_ORDER:
    arr = get_level_map(level_name)
    assert arr.shape == (GRID_SIZE, GRID_SIZE), f"{level_name}: shape attendue (16,16), reçu {arr.shape}"
    assert np.sum(arr == CELL_EXIT) >= 1, f"{level_name}: il faut au moins une sortie"
    assert arr[0, 0] != CELL_WALL, f"{level_name}: la case départ (0,0) ne doit pas être un mur"

print("[OK] les 3 niveaux sont valides.")

[OK] les 3 niveaux sont valides.


In [41]:
print("[INFO] working dir:", os.getcwd())
pth_files = [f for f in os.listdir() if f.endswith(".pth")]
print("[INFO] .pth files:", pth_files)


[INFO] working dir: c:\Users\USER\Downloads\maze_escape_assets
[INFO] .pth files: ['maze_policy_32.pth', 'maze_policy_v3_stable_16x16_fullmap.pth', 'maze_policy_v5_greedy_finetuned_best_greedy.pth', 'maze_policy_v5_greedy_finetuned_best_sample.pth']


In [42]:
N_ACTIONS = 4
ACTION_TO_DELTA = {
    0: (-1, 0),
    1: (1, 0),
    2: (0, -1),
    3: (0, 1),
}
ACTION_NAMES = {0: "UP", 1: "DOWN", 2: "LEFT", 3: "RIGHT"}

R_EXIT = 500.0
R_TRAP = -25.0
R_STEP = -0.05
R_WALL = -0.5
R_COIN = 5.0
R_DEATH = -30.0
R_TIMEOUT = -10.0
DISTANCE_SHAPING = 1.0

FALLBACK_EMOJIS = {
    CELL_FREE: "·",
    CELL_WALL: "🧱",
    CELL_TRAP: "☠",
    CELL_EXIT: "🚪",
    CELL_PLAYER: "🙂",
    CELL_COIN: "🪙",
    CELL_ENEMY: "👾",
}

FALLBACK_COLORS = {
    CELL_FREE: (236, 240, 241),
    CELL_WALL: (52, 73, 94),
    CELL_TRAP: (192, 57, 43),
    CELL_EXIT: (39, 174, 96),
    CELL_PLAYER: (41, 128, 185),
    CELL_COIN: (241, 196, 15),
    CELL_ENEMY: (142, 68, 173),
}


In [43]:
class Enemy:
    def __init__(self, start_r: int, start_c: int, grid: np.ndarray):
        self.r = start_r
        self.c = start_c
        self.circuit = self._build_circuit(start_r, start_c, grid)
        self.idx = 0

    def _build_circuit(self, r: int, c: int, grid: np.ndarray):
        candidates = [
            [(r, c), (r, c + 1), (r + 1, c + 1), (r + 1, c)],
            [(r, c), (r, c - 1), (r + 1, c - 1), (r + 1, c)],
            [(r, c), (r - 1, c), (r - 1, c + 1), (r, c + 1)],
            [(r, c), (r, c + 1), (r, c + 2), (r, c + 3)],
            [(r, c), (r + 1, c), (r + 2, c), (r + 3, c)],
        ]
        for circuit in candidates:
            if all(
                0 <= pr < GRID_SIZE and 0 <= pc < GRID_SIZE and grid[pr, pc] in (CELL_FREE, CELL_ENEMY)
                for pr, pc in circuit
            ):
                return circuit
        return [(r, c)] * 4

    def move(self):
        self.idx = (self.idx + 1) % len(self.circuit)
        self.r, self.c = self.circuit[self.idx]

    @property
    def pos(self) -> Tuple[int, int]:
        return (self.r, self.c)


class MazeEscapeCustom:
    def __init__(self, custom_map: np.ndarray, max_steps: int = 200, start_pos: Tuple[int, int] = (0, 0)):
        self.base_grid = np.array(custom_map, dtype=np.int64).copy()
        assert self.base_grid.shape == (GRID_SIZE, GRID_SIZE)
        self.max_steps = max_steps
        self.start_pos = start_pos
        self.grid_size = GRID_SIZE
        self.n_actions = N_ACTIONS
        self._init_enemies()
        self.max_coins = int(np.sum(self.base_grid == CELL_COIN))
        self.state_dim = N_CELL_TYPES * GRID_SIZE * GRID_SIZE + 7
        self.reset()

    def _init_enemies(self):
        spawns = [(r, c) for r in range(GRID_SIZE) for c in range(GRID_SIZE) if self.base_grid[r, c] == CELL_ENEMY]
        self.enemies = [Enemy(r, c, self.base_grid) for r, c in spawns]

    def _distance_to_exit(self, pos: Tuple[int, int]) -> int:
        exits = np.argwhere(self.base_grid == CELL_EXIT)
        dists = [abs(int(er) - pos[0]) + abs(int(ec) - pos[1]) for er, ec in exits]
        return min(dists) if dists else 0

    def valid_action_mask(self) -> np.ndarray:
        mask = np.ones(self.n_actions, dtype=np.float32)
        for a, (dr, dc) in ACTION_TO_DELTA.items():
            nr, nc = self.pos[0] + dr, self.pos[1] + dc
            if not (0 <= nr < self.grid_size and 0 <= nc < self.grid_size):
                mask[a] = 0.0
            elif self.grid[nr, nc] == CELL_WALL:
                mask[a] = 0.0
        if mask.sum() == 0:
            mask[:] = 1.0
        return mask

    def reset(self):
        self.grid = self.base_grid.copy()
        self.pos = list(self.start_pos)
        self.steps = 0
        self.done = False
        self.total_reward = 0.0
        self.coins_collected = 0
        self.alive = True
        for e in self.enemies:
            e.idx = 0
            e.r, e.c = e.circuit[0]
        return self._get_state()

    def step(self, action: int):
        assert not self.done
        dr, dc = ACTION_TO_DELTA[action]
        nr, nc = self.pos[0] + dr, self.pos[1] + dc
        old_dist = self._distance_to_exit(tuple(self.pos))

        out_of_bounds = not (0 <= nr < self.grid_size and 0 <= nc < self.grid_size)
        hits_wall = (not out_of_bounds) and (self.grid[nr, nc] == CELL_WALL)

        if out_of_bounds or hits_wall:
            reward = R_WALL
        else:
            self.pos = [nr, nc]
            cell = self.grid[nr, nc]
            if cell == CELL_EXIT:
                reward = R_EXIT
                self.done = True
            elif cell == CELL_TRAP:
                reward = R_TRAP
                self.done = True
                self.alive = False
            elif cell == CELL_COIN:
                reward = R_COIN
                self.grid[nr, nc] = CELL_FREE
                self.coins_collected += 1
            else:
                reward = R_STEP

        self.steps += 1

        if not self.done:
            for e in self.enemies:
                e.move()
                if tuple(self.pos) == e.pos:
                    reward += R_DEATH
                    self.done = True
                    self.alive = False
                    break

        new_dist = self._distance_to_exit(tuple(self.pos))
        reward += DISTANCE_SHAPING * (old_dist - new_dist)

        if self.steps >= self.max_steps:
            self.done = True
            reward += R_TIMEOUT

        self.total_reward += reward
        info = {
            "steps": self.steps,
            "pos": tuple(self.pos),
            "coins": self.coins_collected,
            "alive": self.alive,
            "distance_to_exit": new_dist,
            "reached_exit": int(self.alive and self.grid[self.pos[0], self.pos[1]] == CELL_EXIT),
        }
        return self._get_state(), reward, self.done, info

    def _get_state(self) -> np.ndarray:
        grid_view = self.grid.copy().astype(np.int64)
        for e in self.enemies:
            er, ec = e.pos
            if 0 <= er < GRID_SIZE and 0 <= ec < GRID_SIZE:
                grid_view[er, ec] = CELL_ENEMY
        pr, pc = self.pos
        grid_view[pr, pc] = CELL_PLAYER
        one_hot = np.eye(N_CELL_TYPES, dtype=np.float32)[grid_view]
        one_hot = np.transpose(one_hot, (2, 0, 1))
        map_flat = one_hot.reshape(-1)

        global_feats = np.array(
            [
                pr / (GRID_SIZE - 1 + 1e-8),
                pc / (GRID_SIZE - 1 + 1e-8),
                1.0,
                1.0,
                self._distance_to_exit(tuple(self.pos)) / (2 * (GRID_SIZE - 1) + 1e-8),
                1.0 - (self.steps / max(1, self.max_steps)),
                self.coins_collected / max(1, max(1, self.max_coins)),
            ],
            dtype=np.float32,
        )
        return np.concatenate([map_flat, global_feats], axis=0)


In [44]:
class ActorCriticNet(nn.Module):
    def __init__(self, state_dim: int, n_actions: int, hidden_sizes: Tuple[int, int] = (256, 256)):
        super().__init__()
        cnn_input_size = N_CELL_TYPES * GRID_SIZE * GRID_SIZE
        global_size = state_dim - cnn_input_size
        self._cnn_input_size = cnn_input_size

        self.cnn = nn.Sequential(
            nn.Conv2d(N_CELL_TYPES, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        cnn_out = 64 * GRID_SIZE * GRID_SIZE
        in_dim = cnn_out + global_size
        layers = []
        for h in hidden_sizes:
            layers += [nn.Linear(in_dim, h), nn.LayerNorm(h), nn.ReLU()]
            in_dim = h

        self.backbone = nn.Sequential(*layers)
        self.policy_head = nn.Linear(in_dim, n_actions)
        self.value_head = nn.Linear(in_dim, 1)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Conv2d)):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.policy_head.weight, gain=0.01)

    def _split(self, x: torch.Tensor):
        cnn_part = x[:, : self._cnn_input_size].view(-1, N_CELL_TYPES, GRID_SIZE, GRID_SIZE)
        global_part = x[:, self._cnn_input_size :]
        return cnn_part, global_part

    def _apply_action_mask(self, logits: torch.Tensor, action_mask: Optional[np.ndarray] = None):
        if action_mask is None:
            return logits
        mask = torch.tensor(action_mask, dtype=torch.float32, device=logits.device).unsqueeze(0)
        return logits.masked_fill(mask <= 0, -1e9)

    def forward(self, x: torch.Tensor):
        cnn_part, global_part = self._split(x)
        z = torch.cat([self.cnn(cnn_part), global_part], dim=-1)
        z = self.backbone(z)
        return self.policy_head(z), self.value_head(z).squeeze(-1)

    @torch.no_grad()
    def greedy_action(self, state_np: np.ndarray, action_mask: Optional[np.ndarray] = None):
        self.eval()
        x = torch.tensor(state_np, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        logits, _ = self.forward(x)
        logits = self._apply_action_mask(logits, action_mask)
        return torch.argmax(logits, dim=-1).item()

    @torch.no_grad()
    def sample_action(self, state_np: np.ndarray, action_mask: Optional[np.ndarray] = None):
        self.eval()
        x = torch.tensor(state_np, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        logits, _ = self.forward(x)
        logits = self._apply_action_mask(logits, action_mask)
        dist = Categorical(logits=logits)
        return dist.sample().item()

    @torch.no_grad()
    def temperature_action(self, state_np: np.ndarray, temperature: float = 0.3, action_mask: Optional[np.ndarray] = None):
        self.eval()
        x = torch.tensor(state_np, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        logits, _ = self.forward(x)
        logits = self._apply_action_mask(logits, action_mask)
        logits = logits / max(temperature, 1e-6)
        dist = Categorical(logits=logits)
        return dist.sample().item()


In [45]:
class SpriteManager:
    def __init__(self, cell_size: int = 40):
        self.cell_size = cell_size
        self.surfaces: Dict[str, Optional["pygame.Surface"]] = {}

    def load(self):
        if not PYGAME_AVAILABLE:
            return
        for key, path in ASSET_PATHS.items():
            surf = None
            if path and os.path.exists(path):
                try:
                    img = pygame.image.load(path).convert_alpha()
                    surf = pygame.transform.smoothscale(img, (self.cell_size, self.cell_size))
                except Exception:
                    surf = None
            self.surfaces[key] = surf

    def get(self, key: str):
        return self.surfaces.get(key)


class PygameMazeRenderer:
    def __init__(self, env: MazeEscapeCustom, cell_size: int = 40, fps: int = 10, show_grid: bool = True):
        if not PYGAME_AVAILABLE:
            raise RuntimeError("pygame n'est pas installé. Fais: pip install pygame")
        pygame.init()
        pygame.font.init()
        self.env = env
        self.cell_size = cell_size
        self.fps = fps
        self.show_grid = show_grid
        self.info_h = 90
        self.width = GRID_SIZE * cell_size
        self.height = GRID_SIZE * cell_size + self.info_h
        self.screen = pygame.display.set_mode((self.width, self.height))
        pygame.display.set_caption("Maze Escape - custom map visualisation")
        self.clock = pygame.time.Clock()
        self.font = pygame.font.SysFont("arial", max(18, cell_size // 3))
        self.big_font = pygame.font.SysFont("arial", max(24, cell_size // 2), bold=True)
        self.sprite_manager = SpriteManager(cell_size=cell_size)
        self.sprite_manager.load()

    def close(self):
        pygame.quit()

    def pump_events(self) -> bool:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                return False
        return True

    def _draw_tile_fallback(self, cell_value: int, rect):
        color = FALLBACK_COLORS[cell_value]
        pygame.draw.rect(self.screen, color, rect, border_radius=6)
        emoji = FALLBACK_EMOJIS[cell_value]
        txt = self.big_font.render(emoji, True, (20, 20, 20))
        txt_rect = txt.get_rect(center=rect.center)
        self.screen.blit(txt, txt_rect)

    def _cell_key(self, cell_value: int) -> str:
        return {
            CELL_FREE: "free",
            CELL_WALL: "wall",
            CELL_TRAP: "trap",
            CELL_EXIT: "exit",
            CELL_PLAYER: "player",
            CELL_COIN: "coin",
            CELL_ENEMY: "enemy",
        }[cell_value]

    def _composed_grid(self) -> np.ndarray:
        grid = self.env.grid.copy().astype(np.int64)
        for e in self.env.enemies:
            er, ec = e.pos
            if 0 <= er < GRID_SIZE and 0 <= ec < GRID_SIZE:
                grid[er, ec] = CELL_ENEMY
        pr, pc = self.env.pos
        grid[pr, pc] = CELL_PLAYER
        return grid

    def render(self, title: str = "Custom map", extra_lines: Optional[List[str]] = None):
        self.screen.fill((20, 24, 32))
        grid = self._composed_grid()
        for r in range(GRID_SIZE):
            for c in range(GRID_SIZE):
                cell = int(grid[r, c])
                rect = pygame.Rect(c * self.cell_size, r * self.cell_size, self.cell_size, self.cell_size)
                key = self._cell_key(cell)
                sprite = self.sprite_manager.get(key)
                if sprite is not None:
                    self.screen.blit(sprite, rect)
                else:
                    self._draw_tile_fallback(cell, rect)
                if self.show_grid:
                    pygame.draw.rect(self.screen, (70, 70, 80), rect, width=1)

        panel_rect = pygame.Rect(0, GRID_SIZE * self.cell_size, self.width, self.info_h)
        pygame.draw.rect(self.screen, (15, 18, 24), panel_rect)
        header = self.font.render(title, True, (240, 240, 240))
        self.screen.blit(header, (10, GRID_SIZE * self.cell_size + 8))

        base_line = (
            f"steps={self.env.steps}/{self.env.max_steps} | reward={self.env.total_reward:.2f} | "
            f"coins={self.env.coins_collected}/{self.env.max_coins} | pos={tuple(self.env.pos)}"
        )
        line2 = self.font.render(base_line, True, (210, 210, 210))
        self.screen.blit(line2, (10, GRID_SIZE * self.cell_size + 35))

        if extra_lines:
            for i, text in enumerate(extra_lines[:2]):
                surf = self.font.render(text, True, (180, 220, 255))
                self.screen.blit(surf, (10, GRID_SIZE * self.cell_size + 58 + i * 20))

        pygame.display.flip()
        self.clock.tick(self.fps)


In [46]:
def load_model_from_checkpoint(checkpoint_path: str, hidden: Tuple[int, int] = (256, 256)) -> ActorCriticNet:
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint introuvable: {checkpoint_path}")

    dummy_env = MazeEscapeCustom(get_level_map(SELECTED_LEVEL), max_steps=MAX_STEPS)
    model = ActorCriticNet(dummy_env.state_dim, dummy_env.n_actions, hidden_sizes=hidden).to(DEVICE)

    payload = torch.load(checkpoint_path, map_location=DEVICE)
    if isinstance(payload, dict) and "model_state_dict" in payload:
        model.load_state_dict(payload["model_state_dict"], strict=False)
    else:
        model.load_state_dict(payload, strict=False)

    model.eval()
    print("[OK] modèle chargé depuis:", checkpoint_path)
    return model

model = load_model_from_checkpoint(CHECKPOINT_PATH)

[OK] modèle chargé depuis: maze_policy_v5_greedy_finetuned_best_sample.pth


In [47]:
@torch.no_grad()
def evaluate_on_level(model: ActorCriticNet, level_name: str, strategy: str = "sample", temperature: float = 0.3, episodes: int = 20):
    rewards, successes = [], []
    for _ in range(episodes):
        env = MazeEscapeCustom(get_level_map(level_name), max_steps=MAX_STEPS)
        state = env.reset()
        while not env.done:
            mask = env.valid_action_mask()
            if strategy == "greedy":
                action = model.greedy_action(state, action_mask=mask)
            elif strategy == "temperature":
                action = model.temperature_action(state, temperature=temperature, action_mask=mask)
            else:
                action = model.sample_action(state, action_mask=mask)
            state, _, _, _ = env.step(action)
        rewards.append(env.total_reward)
        successes.append(int(env.alive and env.grid[env.pos[0], env.pos[1]] == CELL_EXIT))
    print(f"[{level_name.upper()}] strategy={strategy} | avg_reward={np.mean(rewards):.2f} | success={np.mean(successes)*100:.1f}%")

def evaluate_all_levels(model: ActorCriticNet, strategy: str = "sample", temperature: float = 0.3, episodes: int = 20):
    print(f"=== Evaluation sur les 3 niveaux | strategy={strategy} ===")
    for level_name in LEVEL_ORDER:
        evaluate_on_level(model, level_name=level_name, strategy=strategy, temperature=temperature, episodes=episodes)

evaluate_all_levels(model, strategy="sample")
evaluate_all_levels(model, strategy="greedy")
evaluate_all_levels(model, strategy="temperature", temperature=TEMPERATURE)

=== Evaluation sur les 3 niveaux | strategy=sample ===
[EASY] strategy=sample | avg_reward=204.25 | success=40.0%
[MEDIUM] strategy=sample | avg_reward=155.14 | success=30.0%
[HARD] strategy=sample | avg_reward=-16.54 | success=0.0%
=== Evaluation sur les 3 niveaux | strategy=greedy ===
[EASY] strategy=greedy | avg_reward=528.55 | success=100.0%
[MEDIUM] strategy=greedy | avg_reward=528.55 | success=100.0%
[HARD] strategy=greedy | avg_reward=-14.00 | success=0.0%
=== Evaluation sur les 3 niveaux | strategy=temperature ===
[EASY] strategy=temperature | avg_reward=499.10 | success=95.0%
[MEDIUM] strategy=temperature | avg_reward=205.73 | success=40.0%
[HARD] strategy=temperature | avg_reward=-14.08 | success=0.0%


In [66]:
def watch_level(model: ActorCriticNet, level_name: str, strategy: str = "greedy", temperature: float = 0.3, fps: int = 8, pause_seconds: float = 2.0):
    if not PYGAME_AVAILABLE:
        raise RuntimeError("pygame n'est pas installé. Fais: pip install pygame")

    env = MazeEscapeCustom(get_level_map(level_name), max_steps=MAX_STEPS)
    state = env.reset()
    renderer = PygameMazeRenderer(env, cell_size=CELL_SIZE, fps=fps)

    try:
        running = True
        while running:
            running = renderer.pump_events()
            if env.done:
                status = "SUCCESS" if (env.alive and env.grid[env.pos[0], env.pos[1]] == CELL_EXIT) else "FAILED"
                renderer.render(
                    title=f"{level_name.upper()} - {status}",
                    extra_lines=[
                        f"strategy={strategy}",
                        "Ferme la fenêtre pour quitter."
                    ],
                )
                time.sleep(pause_seconds)
                break

            mask = env.valid_action_mask()

            if strategy == "greedy":
                action = model.greedy_action(state, action_mask=mask)
            elif strategy == "temperature":
                action = model.temperature_action(state, temperature=temperature, action_mask=mask)
            else:
                action = model.sample_action(state, action_mask=mask)

            state, reward, done, info = env.step(action)
            renderer.render(
                title=f"{level_name.upper()} - visualisation du modèle",
                extra_lines=[
                    f"strategy={strategy} | action={ACTION_NAMES[action]}",
                    f"step_reward={reward:.2f}"
                ],
            )
        return True
    finally:
        renderer.close()


def watch_all_levels(model: ActorCriticNet, strategy: str = "greedy", temperature: float = 0.3, fps: int = 8, pause_between_levels: float = 1.0):
    if not PYGAME_AVAILABLE:
        raise RuntimeError("pygame n'est pas installé. Fais: pip install pygame")

    for level_name in LEVEL_ORDER:
        env = MazeEscapeCustom(get_level_map(level_name), max_steps=MAX_STEPS)
        state = env.reset()
        renderer = PygameMazeRenderer(env, cell_size=CELL_SIZE, fps=fps)

        try:
            running = True
            while running:
                running = renderer.pump_events()
                if not running:
                    return

                if env.done:
                    status = "SUCCESS" if (env.alive and env.grid[env.pos[0], env.pos[1]] == CELL_EXIT) else "FAILED"
                    renderer.render(
                        title=f"{level_name.upper()} - {status}",
                        extra_lines=[
                            f"strategy={strategy}",
                            f"Niveau suivant dans {pause_between_levels:.1f}s"
                        ],
                    )
                    time.sleep(pause_between_levels)
                    break

                mask = env.valid_action_mask()

                if strategy == "greedy":
                    action = model.greedy_action(state, action_mask=mask)
                elif strategy == "temperature":
                    action = model.temperature_action(state, temperature=temperature, action_mask=mask)
                else:
                    action = model.sample_action(state, action_mask=mask)

                state, reward, done, info = env.step(action)
                renderer.render(
                    title=f"{level_name.upper()} - visualisation du modèle",
                    extra_lines=[
                        f"strategy={strategy} | action={ACTION_NAMES[action]}",
                        f"step_reward={reward:.2f}"
                    ],
                )
        finally:
            renderer.close()


# =========================
# Paramètres de visualisation
# =========================
PLAY_STRATEGY = "greedy"
TEMPERATURE = 0,3  # ignoré en mode greedy
FPS = 3

# 1) Pour voir seulement un niveau :
# watch_level(model, level_name=SELECTED_LEVEL, strategy=PLAY_STRATEGY, temperature=TEMPERATURE, fps=FPS)

# 2) Pour voir les 3 niveaux à la file :
watch_all_levels(model, strategy=PLAY_STRATEGY, temperature=TEMPERATURE, fps=FPS)

## Optionnel : jouer vous-même sur le niveau sélectionné

Commandes :
- flèches ou `WASD`
- `ZQSD` marche aussi
- ferme la fenêtre pour quitter

Le mode manuel utilise le niveau défini par `SELECTED_LEVEL`.

In [49]:
def play_manual_level(level_name: str = None, fps: int = 10):
    if not PYGAME_AVAILABLE:
        raise RuntimeError("pygame n'est pas installé. Fais: pip install pygame")
    if level_name is None:
        level_name = SELECTED_LEVEL

    env = MazeEscapeCustom(get_level_map(level_name), max_steps=MAX_STEPS)
    env.reset()
    renderer = PygameMazeRenderer(env, cell_size=CELL_SIZE, fps=fps)

    key_to_action = {
        pygame.K_UP: 0, pygame.K_w: 0, pygame.K_z: 0,
        pygame.K_DOWN: 1, pygame.K_s: 1,
        pygame.K_LEFT: 2, pygame.K_a: 2, pygame.K_q: 2,
        pygame.K_RIGHT: 3, pygame.K_d: 3,
    }

    try:
        running = True
        last_action = None
        while running:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False
                elif event.type == pygame.KEYDOWN:
                    if not env.done and event.key in key_to_action:
                        action = key_to_action[event.key]
                        env.step(action)
                        last_action = ACTION_NAMES[action]

            status = "RUNNING"
            if env.done:
                status = "SUCCESS" if env.alive and env.grid[env.pos[0], env.pos[1]] == CELL_EXIT else "FAILED"

            renderer.render(
                title=f"Mode manuel - {level_name.upper()} - {status}",
                extra_lines=["Arrows / WASD / ZQSD", f"last_action={last_action}"],
            )
    finally:
        renderer.close()

# Décommente pour jouer manuellement sur le niveau sélectionné :
# play_manual_level(SELECTED_LEVEL)

## Conseils

- Le notebook contient maintenant **3 niveaux** : `easy`, `medium`, `hard`.
- `SELECTED_LEVEL` sert pour un test ciblé sur un seul niveau.
- `watch_all_levels(...)` affiche les **3 niveaux à la file**.
- Le notebook cherche automatiquement le modèle **`maze_policy_v5_greedy_finetuned_best_sample.pth`**.
- Avec votre meilleur modèle actuel, **`sample`** est le meilleur point de départ.
- Si `greedy` échoue mais `sample` marche, c'est normal avec ce modèle.
- Tu peux aussi tester `temperature=0.2` ou `0.3`.